In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import numpy as np
from nltk.corpus import brown

vocab = np.unique(brown.words()).reshape(-1, 1)
words = brown.words()

""" Hyperparameters """
vocab_size = len(vocab)
window = 2

# print("Vocabulary Size: ", vocab_size)
# print("Vocabulary: ", vocab)

### Creating word encoding

In [ ]:
# from sklearn.preprocessing import OneHotEncoder
# 
# 
# enc = OneHotEncoder(handle_unknown='ignore') # one hot encoding for the entire vocabulary 
# enc.fit(vocab)

# word = 'and'
# categories = enc.categories_
# test_enc = enc.transform([[word]]).toarray()
# test_word = enc.inverse_transform(test_enc)

# print("Categories: ", categories)
# print(f"One-Hot Vector for '{word}': ", test_enc)
# print("One-Hot Vector Shape: ", test_enc.shape)
# print("Mapping One-Hot Vector to Word: ", test_word)

In [ ]:
# """ 
# The below approach is highly inefficient because we use one-hot encoding.
# Since the vocab size is ~50,000 words, we have 50,000 sparse one-hot vectors 
#     that are very inefficient to perform computations with and have no semantic meaning. 
# Instead, we could better represent words using embeddings.
# """

# X, y = [], []
# for i in range(len(words)):
#     center, context = [words[i]], []

#     for j in range(window): # words before the center
#         if (i + j) - window >= 0:
#             context.append(words[(i + j) - window])
    
#     for j in range(window): # words after the center
#         if i + j + 1 < len(words):
#             context.append(words[i + j + 1])
    
#     X.append(center)
#     y.append(context)

In [ ]:
wtoi, itow = {}, {}
for i in range(len(vocab)): # mapping words to indicies
    wtoi[vocab[i].item()] = i
    itow[i] = vocab[i].item()

X, y = [], []
for i in range(window, len(words) - window):
    center = wtoi[words[i]]
    for j in range(i - window, i + window + 1):
        if j != i:
            context = wtoi[words[j]]
            X.append(center)
            y.append(context)

In [ ]:
from sklearn.model_selection import train_test_split

batch_size = 64
class SkipgramDataset(Dataset):
    def __init__(self, center, context):
        self.center = torch.tensor(center)
        self.context = torch.tensor(context)

    def __len__(self):
        return len(self.center)

    def __getitem__(self, idx):
        return self.center[idx], self.context[idx]
    
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

training_set = SkipgramDataset(X_train, y_train)
testing_set = SkipgramDataset(X_test, y_test)
train_dataloader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(testing_set, batch_size=batch_size, shuffle=True)

In [ ]:
# for i in range(10): # visualizing some training datapoints
#     print(X_train[i], y_train[i])

In [ ]:
embedding_dim = 2

class Skipgram(nn.Module):
    def __init__(self):
        super().__init__() # neural network inherits from nn.Module

        """ skipgram network layers """
        # self.model = nn.Sequential(
        #     nn.Linear(vocab_size, hidden),
        #     nn.Linear(vocab_size, vocab_size * 4)
        # )
        self.model = nn.Sequential(
            nn.Embedding(vocab_size, embedding_dim),
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, vocab_size)
        )

    """ forward pass of data """
    def forward(self, x):
        logits = self.model(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Skipgram().to(device)
print(f"{model} \n Using {device} device")

In [ ]:
criterion = nn.CrossEntropyLoss() # cross entropy loss
optimizer = optim.SGD(model.parameters(), lr=0.01) # stochastic gradient descent algorithm

epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad() # zero gradients
    for center, context in train_dataloader:
        center = center.to(device)
        context = context.to(device)
        out = model(center) # model predicts on training set
        loss = criterion(out, context) # calculating overall loss

        loss.backward() # computing gradients from loss function
        optimizer.step() # update the weights             

    print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}') 

In [ ]:
word = 'The'
test_data = torch.tensor([wtoi[word]], dtype=torch.long).to(device)

with torch.no_grad():
    prediction = model(test_data)
    pred_idx = torch.argmax(prediction, dim=1).item()

print("Predicted context word:", itow[pred_idx])